[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/12-observability.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 12. Observability

Each span names three identities: the customer, the Harbor Jar seller, and the tool. The chili-oil question from tutorial 2 runs again so the same work has a trace. Nothing secret belongs in the JSON.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Spans

Importing `tutorial.common.trace` turns spans on inside `run_agent`. The catalog and the topic lookup are the earlier tools.

In [ ]:
import json

import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.trace  # records customer, agent, and tool spans

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_db()
result = run_agent(
    "Do you have Calabrian chili oil, and can you ship a jar to Ohio?",
    system_text(),
    trace_id="trace-chili-ohio",
)
print("ANSWER:", result["text"])
print(json.dumps(result["spans"], indent=2, sort_keys=True))
blob = json.dumps(result["spans"])
check(result["stopped"] == "final", "the traced question finished")
check(len(result["spans"]) >= 2, "the turn span and at least one tool span were recorded")
check(all(span["user"]["id"] == "customer" for span in result["spans"]), "every span names the customer")
check(all(span["agent"]["id"] == "harbor-jar-concierge" for span in result["spans"]), "every span names the seller")
check(any(span["actor"]["kind"] == "tool" for span in result["spans"]), "a tool has its own actor")
check("sk-" not in blob and "canary" not in blob.lower(), "the trace does not carry a secret")